# 👁️ AI Smart Camera v0.2.2: Security & Correctness Hotfix

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tahadeab-ai/ai-smart-camera/blob/main/notebooks/01_ai_smart_camera_colab.ipynb)
[![GitHub License](https://img.shields.io/badge/License-MIT-blue.svg)](https://github.com/tahadeab-ai/ai-smart-camera/blob/main/LICENSE)
[![FastAPI](https://img.shields.io/badge/Backend-FastAPI-009688.svg)](https://fastapi.tiangolo.com/)
[![React 18](https://img.shields.io/badge/Frontend-React%2018%20%2B%20TypeScript-61dafb.svg)](https://react.dev/)
[![YOLO](https://img.shields.io/badge/Model-YOLOv8%20%7C%20YOLO11-orange.svg)](https://docs.ultralytics.com/)
[![Supervision](https://img.shields.io/badge/Vision-Supervision-green.svg)](https://supervision.roboflow.com/)

Welcome to the official **Google Colab Interactive Execution Environment** for **AI Smart Camera v0.2.2**.

### v0.2.2 Hardening & Security Highlights
- 🔒 **Strict Ticket-Only WebSocket Authentication**: Permanent token query parameter fallback completely eliminated; single-use, 30s TTL cryptographically secure tickets required.
- 🛡️ **Strict WebSocket Origin Policy**: Wildcard CORS origins rejected for authenticated deployments; Origin headers strictly enforced by default.
- 📸 **Real Hardware Camera Switching**: Full 9-step asynchronous hardware transition in `useCamera`, pausing frames, cleanly terminating old tracks, clearing references, acquiring new stream, and waiting for video readiness.
- 🛑 **ASGI Body Limits & Pre-Decode Pillow Inspection**: Early ASGI-level body rejection (20MB) and Pillow header inspection verifying image dimensions and pixel count (<= 16MP) before allocating raster OpenCV buffers.
- ⏱️ **WebSocket Frame Size Bounds**: Maximum WebSocket frame size enforced (2MB) with JPEG magic byte validation prior to inference.
- 🎛️ **Rate Limiter Memory Bounds & Trusted Proxy**: Bounded storage capacity (`max_tracked_clients=10000`) with stale entry eviction; `X-Forwarded-For` parsed only from configured `trusted_proxies`.
- 📦 **Strict Dependency Locking**: Enforced `npm ci` without unpinned fallbacks in Docker and CI workflows.

---

## 1. Environment & Hardware Inspection
Inspect the Colab runtime environment, including CPU, RAM, and CUDA GPU acceleration (with automatic CPU fallback).

In [ ]:
import sys
import os
import platform
import psutil
import torch

print('=' * 60)
print('AI SMART CAMERA v0.2.2 - HARDWARE & ENVIRONMENT AUDIT')
print('=' * 60)
print(f'Python Version   : {platform.python_version()} ({platform.python_implementation()})')
print(f'Operating System : {platform.system()} {platform.release()}')
print(f'CPU Logical Cores: {psutil.cpu_count(logical=True)}')
ram_gb = psutil.virtual_memory().total / (1024 ** 3)
print(f'System Memory    : {ram_gb:.2f} GB RAM')

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    device_mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f'CUDA Accelerator : {device_name} ({device_mem:.2f} GB VRAM)')
    DEFAULT_DEVICE = 'cuda'
else:
    print('CUDA Accelerator : Not detected. Using CPU execution mode.')
    DEFAULT_DEVICE = 'cpu'
print('=' * 60)


## 2. Clone Repository
Clone the permanent GitHub source of truth (`tahadeab-ai/ai-smart-camera`).

In [ ]:
%cd /content
!rm -rf ai-smart-camera
!git clone https://github.com/tahadeab-ai/ai-smart-camera.git
%cd ai-smart-camera
!git status


## 3. Install Pinned Python Dependencies
Install verified Python packages from `requirements-dev.txt` for development, testing, and benchmarking.

In [ ]:
%cd /content/ai-smart-camera
!pip install --upgrade pip
!pip install -r requirements-dev.txt


## 4. Model Checksum & Manifest Verification
Download and verify YOLO model weights against the SHA-256 cryptographic hashes specified in `models/manifest.yaml`.

In [ ]:
%cd /content/ai-smart-camera
from app.core.model_loader import ModelLoader

loader = ModelLoader()
print('Verifying models against manifest checksums:')
for model_key in ['yolov8n', 'yolo11n']:
    meta = loader.get_model_metadata(model_key)
    path = loader.ensure_model_available(model_key)
    sha = loader._calculate_sha256(path)
    assert sha == meta['sha256'], f'Checksum mismatch for {model_key}!'
    print(f'  [PASS] {model_key}: {path.name} (SHA-256: {sha[:16]}...)')


## 5. Automated Python Test Suite Execution
Run the complete unit and integration test suite with test coverage reporting.

In [ ]:
%cd /content/ai-smart-camera
!pytest -v tests/ --cov=app --cov-report=term-missing


## 6. Compile React 18 + TypeScript Frontend (`npm ci`)
Install deterministic npm dependencies and compile the production single-page application into `frontend/dist`.

In [ ]:
%cd /content/ai-smart-camera/frontend
!npm ci
!npm test
!npm run build
!ls -la /content/ai-smart-camera/frontend/dist


## 7. Run Reproducible Hardware Benchmarks
Execute the automated benchmarking framework measuring P50/P95 latency, statistical variance, and throughput FPS on this Colab runtime.

In [ ]:
%cd /content/ai-smart-camera
!python benchmarks/benchmark.py --model yolov8n --device {DEFAULT_DEVICE} --runs 30
!cat benchmarks/benchmark_report.md


## 8. Generate Random Deployment Access Token
Generate a cryptographically secure random session token. Stored securely in a private local file to prevent leakage in logs or screen captures.

In [ ]:
import secrets
from pathlib import Path

DEPLOY_TOKEN = secrets.token_urlsafe(24)
token_file = Path('/content/.access_token')
token_file.write_text(DEPLOY_TOKEN.strip())
print('=' * 65)
print('Session access token generated and saved to /content/.access_token')
print('To view your token securely, inspect /content/.access_token')
print('=' * 65)


## 9. Launch Cloudflare Quick Tunnel & FastAPI Backend
Dynamically initialize Cloudflare Quick Tunnel to capture the exact public origin, inject it into CORS_ORIGINS, and launch the hardened FastAPI backend.

In [ ]:
%cd /content/ai-smart-camera
import json
import os
import re
import subprocess
import sys
import time
import urllib.request

# Install cloudflared if not present
!which cloudflared || (wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb)

# Terminate prior processes
!pkill -f 'cloudflared' || true
!pkill -f 'app.main' || true

# 1. Start Cloudflare Tunnel
tunnel_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:7860'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
for _ in range(40):
    line = tunnel_proc.stdout.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

if not public_url:
    raise RuntimeError('Failed to acquire Cloudflare tunnel URL.')

# 2. Configure Dynamic CORS Origin and Environment
cors_list = ['http://localhost:7860', 'http://127.0.0.1:7860', public_url]
env = dict(os.environ)
env['CORS_ORIGINS'] = json.dumps(cors_list)
env['ACCESS_TOKEN'] = DEPLOY_TOKEN

# 3. Launch FastAPI backend with injected origin
server_proc = subprocess.Popen(
    [
        sys.executable, '-m', 'app.main',
        '--host', '0.0.0.0',
        '--port', '7860',
        '--model', 'yolov8n',
        '--device', DEFAULT_DEVICE,
    ],
    env=env,
    stdout=open('/content/server.log', 'w'),
    stderr=subprocess.STDOUT,
)

print('Waiting for FastAPI server to initialize...')
for _ in range(30):
    try:
        with urllib.request.urlopen('http://localhost:7860/api/v1/health/live') as res:
            if res.status == 200:
                print('FastAPI server initialized successfully.')
                break
    except Exception:
        time.sleep(1)
else:
    print('Warning: Server startup timed out. Check /content/server.log')

print('=' * 65)
print(f'PUBLIC HTTPS APPLICATION URL : {public_url}')
print(f'WEBSOCKET WSS ENDPOINT       : {public_url.replace("https:", "wss:")}/api/v1/stream')
print(f'DYNAMIC CORS ORIGIN          : {public_url}')
print('=' * 65)
print('Instructions:')
print('1. Open the Public HTTPS URL in your mobile or desktop browser.')
print('2. Retrieve your token by inspecting /content/.access_token')
print('3. Click Token in the top navbar and submit your session token.')
print('4. Click Live Camera, allow camera access, and test real-time object detection.')
print('=' * 65)


## 10. Clean Server Shutdown
Safely terminate all background processes and clean up runtime resources when finished.

In [ ]:
!pkill -f 'cloudflared' || true
!pkill -f 'app.main' || true
print('All background processes stopped. Runtime clean.')
